# 📄 RAG PDF Bot — AI Document Q&A
> Query any PDF using LangChain + FAISS + HuggingFace (Google Colab)

**Pipeline:** PDF → Chunking → Embeddings → FAISS → Retrieval → LLM → Answer

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Fix Colab Dependencies

In [ ]:
# Fix requests version conflict in Colab
!pip install requests==2.32.4 --force-reinstall --quiet

import requests
print(f"✅ requests version: {requests.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 15.7 MB/s eta 0:00:00
✅ requests version: 2.32.4


##Install Dependencies

In [1]:
!pip install --upgrade pip

!pip install \
  numpy==1.26.4 \
  "tenacity>=8.1.0,<9.0.0" \
  langchain==0.2.16 \
  langchain-community==0.2.16 \
  langchain-core==0.2.43 \
  langchain-huggingface \
  faiss-cpu \
  sentence-transformers \
  transformers \
  pypdf \
  --quiet

print("✅ All dependencies installed")

✅ All dependencies installed


##Load & Chunk the PDF

> **Before running:** Upload your PDF to Google Drive then add Path

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

#config
PDF_PATH = "/content/drive/MyDrive/IKS_4001_MII_Lecture_0.pdf"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# Verify file exists
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"❌ PDF not found at: {PDF_PATH}\n"
                            f"   Files in /content: {os.listdir('/content')}")

# Load PDF
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

if len(docs) == 0:
    raise ValueError("❌ PDF loaded 0 pages. Try re-uploading.")

print(f"✅ Loaded {len(docs)} page(s)")
print(f"   Preview: {docs[0].page_content[:150].strip()}...")

# Chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(docs)

if len(chunks) == 0:
    raise ValueError("❌ No text chunks extracted. Use a text-based PDF (not scanned).")

print(f"✅ Created {len(chunks)} chunk(s)")

✅ Loaded 18 page(s)
   Preview: L#00
Subject Information
Mathematics In India
(IKS 4001)
Dr. Sabita Mali
Assistant Professor, Dept. of ECE,
Faculty of Engineering and Technology (ITE...
✅ Created 30 chunk(s)


##Build FAISS Vector Store

In [3]:
import warnings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL} ...")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # suppress non-critical load warnings
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

db = FAISS.from_documents(chunks, embeddings)
print("✅ FAISS vector store ready")

Loading embedding model: sentence-transformers/all-mpnet-base-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS vector store ready


##Load LLM & Build RAG Chain

> `flan-t5-base` is a **text-to-text** model — use `text-generation` in new version

In [21]:
import warnings
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_huggingface import HuggingFacePipeline
from langchain.chains import RetrievalQA

MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llmodel = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"Loading LLM: {LLM_MODEL} ...")

# flan-t5 is a seq2seq model → use text-generation in new version
pipe = pipeline(
    "text-generation",
    model=llmodel,
    tokenizer=tokenizer,
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=pipe)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    return_source_documents=True,
    retriever=db.as_retriever(search_kwargs={"k": 4})
)

print("✅ RAG chain ready")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

Loading LLM: google/flan-t5-base ...
✅ RAG chain ready


##Ask Questions

Use `ask(query)` to query the document. Only the final answer is printed.

In [25]:
def ask(query: str):
    result = qa_chain.invoke({"query": query})
    answer = " ".join(result["result"].split())
    print("\n🧠 Answer:\n", answer)
    print("\n📚 Sources:")
    for doc in result["source_documents"]:
        print("-", doc.metadata.get("source", "unknown"))
    return answer

In [26]:
ask("Summarize this document in a few sentences.")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🧠 Answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. suggests a practical understanding of counting, measurement, and geometry (pp. 35-36). The precise construction of right angles may have used a method involving intersecting circles rather than the Pythagorean theorem. Summary • The Indus Valley Civilisation was a highly developed urban civilisation known for its city planning, sanitation, trade system, and practical knowledge . It laid the foundation for later cultures in the Indian subcontinent. with no clear signs of large palaces or temples. People wore cotton clothes and lived an organized urban life. 6. Script and Religion The Indus script is still undeciphered , so much of their beliefs remain unknown. However, seals suggest worship of nature, animals, and possibly early forms of Shiva (Pashupati) . 7. Mathematical Implications : The uniformity of weights a

"Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. suggests a practical understanding of counting, measurement, and geometry (pp. 35-36). The precise construction of right angles may have used a method involving intersecting circles rather than the Pythagorean theorem. Summary • The Indus Valley Civilisation was a highly developed urban civilisation known for its city planning, sanitation, trade system, and practical knowledge . It laid the foundation for later cultures in the Indian subcontinent. with no clear signs of large palaces or temples. People wore cotton clothes and lived an organized urban life. 6. Script and Religion The Indus script is still undeciphered , so much of their beliefs remain unknown. However, seals suggest worship of nature, animals, and possibly early forms of Shiva (Pashupati) . 7. Mathematical Implications : The uniformity of weights and measures

In [24]:
ask("What are the key topics covered in this document?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🧠 Answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. • The Vedas (especially the Yajur Veda ) • Vedangas , particularly the Ś ulba S ū tras These texts contain rules and methods related to numbers, measurements, and geometry. 3. Economy and Trade The people were mainly farmers and traders . They grew wheat, barley, cotton, and traded with distant regions like Mesopotamia . Standardized weights and measures were used for trade. 4. Technology and Crafts The civilisation was skilled in pottery, bead-making, metallurgy, and seal carving . Seals made of steatite were used for trade and identification. 5. Social Life and Culture Evidence suggests a peaceful society with no clear signs of large palaces or Typical Exam Questions • Explain the role of the Indus Valley Civilisation in early mathematics. • Why was mathematics closely linked to rituals in the Vedic period? • Ho

"Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. • The Vedas (especially the Yajur Veda ) • Vedangas , particularly the Ś ulba S ū tras These texts contain rules and methods related to numbers, measurements, and geometry. 3. Economy and Trade The people were mainly farmers and traders . They grew wheat, barley, cotton, and traded with distant regions like Mesopotamia . Standardized weights and measures were used for trade. 4. Technology and Crafts The civilisation was skilled in pottery, bead-making, metallurgy, and seal carving . Seals made of steatite were used for trade and identification. 5. Social Life and Culture Evidence suggests a peaceful society with no clear signs of large palaces or Typical Exam Questions • Explain the role of the Indus Valley Civilisation in early mathematics. • Why was mathematics closely linked to rituals in the Vedic period? • How did oral 

In [27]:
ask("What does the document say about mathematics?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🧠 Answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Overall Significance of Chapter 1 Chapter 1 shows that: • Indian mathematics has very deep roots • It developed long before formal texts • Culture and language played a major role • Mathematics was closely connected to daily life and religion This chapter prepares the reader to understand why later Indian mathematics developed in a unique and powerful way , different from Greek or European traditions. Chapter 1: Background – Culture and Language • Mathematics in the Indus Valley Civilisation • Mathematics in the Vedic Period • Importance of the Oral Tradition • Language, Grammar, and Mathematics • Cultural Foundations of Indian Mathematics Summary • Indus Valley → geometry & measurement • Vedic period → ritual mathematics • Oral tradition → sutras, no symbols • Language → procedural thinking • P āṇ ini → rule-base

"Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Overall Significance of Chapter 1 Chapter 1 shows that: • Indian mathematics has very deep roots • It developed long before formal texts • Culture and language played a major role • Mathematics was closely connected to daily life and religion This chapter prepares the reader to understand why later Indian mathematics developed in a unique and powerful way , different from Greek or European traditions. Chapter 1: Background – Culture and Language • Mathematics in the Indus Valley Civilisation • Mathematics in the Vedic Period • Importance of the Oral Tradition • Language, Grammar, and Mathematics • Cultural Foundations of Indian Mathematics Summary • Indus Valley → geometry & measurement • Vedic period → ritual mathematics • Oral tradition → sutras, no symbols • Language → procedural thinking • P āṇ ini → rule-based abstracti